# Image Branch — ResNet50 Baseline and Grad-CAM

[![GitHub](https://img.shields.io/badge/GitHub-manasdutta04%2Fmultimodal--fake--news--detector-181717?logo=github&logoColor=white)](https://github.com/manasdutta04/multimodal-fake-news-detector)

This notebook implements the **image-only** path of *Multimodal Fake News Detector with Explainability*: binary real/fake classification from Fakeddit post images, plus region-level attribution via Grad-CAM.

It is the second trained component of the system. Downstream fusion reuses this ResNet50 encoder (and Module 01 DistilBERT) as unimodal baselines.

> **Note.** Research prototype — not a deployed fact-checker. Visual cues alone do not verify claims against an external knowledge base.

**Scope**
- Dataset: Fakeddit multimodal-only TSVs (`image_url`, `2_way_label`, `id`)
- Model: ImageNet-pretrained ResNet50; freeze early blocks; fine-tune `layer4` + classification head
- Metrics: accuracy, per-class precision/recall/F1, confusion matrices on official val/test (rows with a successfully downloaded image)
- Explainability: Grad-CAM on the last convolutional block

**Inputs.** Same Drive folder as Module 01 (`DATA_DIR` with the three TSVs). Images are downloaded into an on-Drive cache so re-runs do not refetch every URL.

**Prerequisite.** Module 01 text metrics/checkpoint may already exist under `DATA_DIR/checkpoints/`; this notebook does not require loading them.


## Environment

Requires a GPU runtime for ResNet50 fine-tuning. Image download can run on CPU but is I/O-bound.

> **Note.** Colab: **Runtime → Change runtime type → GPU** (T4 or better).


## Dependencies


In [ ]:
# Image training + Grad-CAM
%pip install -q scikit-learn pandas numpy matplotlib seaborn pillow requests tqdm
%pip install -q "pytorch-grad-cam>=1.5.0"

# PyTorch / torchvision with CUDA are provided by Colab/Kaggle GPU images.


> **Note.** After installing on a fresh runtime, restart the session once, then continue from imports.


In [ ]:
# --- Standard library / numerics ---
import os
import io
import time
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


## Dataset path

Point `DATA_DIR` at the Drive folder that already holds the three multimodal TSVs (typically `/content/drive/MyDrive/dataset`). Image files are cached under `DATA_DIR/image_cache/`.


In [ ]:
# True  → mount Google Drive (Colab)
# False → local / Kaggle path in DATA_DIR
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/dataset"
else:
    # Kaggle: "/kaggle/input/<dataset>"  |  session upload: "/content/dataset"
    DATA_DIR = "/content/dataset"

TRAIN_PATH = os.path.join(DATA_DIR, "multimodal_train.tsv")
VAL_PATH = os.path.join(DATA_DIR, "multimodal_validate.tsv")
TEST_PATH = os.path.join(DATA_DIR, "multimodal_test_public.tsv")
IMAGE_CACHE = os.path.join(DATA_DIR, "image_cache")
os.makedirs(IMAGE_CACHE, exist_ok=True)

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    assert os.path.exists(p), f"Missing file: {p}"
print("All split files found.")
print("Image cache:", IMAGE_CACHE)


## Load official splits

Use Fakeddit’s published train / validate / test files. Drop rows with missing `image_url` before download.

`SUBSAMPLE_*` caps how many rows enter the download + train pipeline on hosted GPUs. Set to `None` for full-split image training when preparing report numbers.


In [ ]:
LABEL_COL = "2_way_label"  # 0 = fake, 1 = real
ID_COL = "id"
URL_COL = "image_url"

# None = use full split (after URL filter). Integer = stratified subsample.
SUBSAMPLE_TRAIN = 8_000
SUBSAMPLE_VAL = 2_000
SUBSAMPLE_TEST = 2_000


def load_split(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", low_memory=False)
    needed = [ID_COL, URL_COL, LABEL_COL, "hasImage"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")
    return df


def filter_urls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out[URL_COL] = out[URL_COL].astype(str).str.strip()
    out = out[out[URL_COL].notna() & (out[URL_COL] != "") & (out[URL_COL].str.lower() != "nan")]
    out = out[out[URL_COL].str.startswith(("http://", "https://"))]
    return out.reset_index(drop=True)


def stratified_subsample(df: pd.DataFrame, n):
    if n is None or n >= len(df):
        return df.reset_index(drop=True)
    parts = []
    per_class = max(1, n // 2)
    for _, group in df.groupby(LABEL_COL):
        parts.append(group.sample(n=min(len(group), per_class), random_state=SEED))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=SEED)
    return out.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


train_df = stratified_subsample(filter_urls(load_split(TRAIN_PATH)), SUBSAMPLE_TRAIN)
val_df = stratified_subsample(filter_urls(load_split(VAL_PATH)), SUBSAMPLE_VAL)
test_df = stratified_subsample(filter_urls(load_split(TEST_PATH)), SUBSAMPLE_TEST)

print(f"After URL filter + subsample → train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
print("Train label counts:\n", train_df[LABEL_COL].value_counts().sort_index())


## Download and validate images

Fetch each URL into `image_cache/{id}.jpg`. Skip broken links, non-image payloads, and tiny/corrupt files. Existing cache hits are reused.

> **Note.** Many Reddit/Imgur preview links expire or rate-limit. Expect a non-trivial failure rate; training uses only successful downloads.


In [ ]:
DOWNLOAD_WORKERS = 16
REQUEST_TIMEOUT = 12
MIN_SIDE_PX = 32
USER_AGENT = (
    "Mozilla/5.0 (compatible; MultimodalFakeNewsDetector/0.1; "
    "+https://github.com/manasdutta04/multimodal-fake-news-detector)"
)


def cache_path_for(sample_id: str) -> str:
    safe = str(sample_id).replace("/", "_")
    return os.path.join(IMAGE_CACHE, f"{safe}.jpg")


def is_valid_image_file(path: str) -> bool:
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im = im.convert("RGB")
            w, h = im.size
            return w >= MIN_SIDE_PX and h >= MIN_SIDE_PX
    except Exception:
        return False


def download_one(sample_id: str, url: str) -> tuple[str, bool, str]:
    path = cache_path_for(sample_id)
    if os.path.exists(path) and is_valid_image_file(path):
        return sample_id, True, "cached"
    try:
        resp = requests.get(
            url,
            timeout=REQUEST_TIMEOUT,
            headers={"User-Agent": USER_AGENT},
            stream=True,
        )
        if resp.status_code != 200:
            return sample_id, False, f"http_{resp.status_code}"
        data = resp.content
        if not data or len(data) < 100:
            return sample_id, False, "empty"
        with Image.open(io.BytesIO(data)) as im:
            im = im.convert("RGB")
            w, h = im.size
            if w < MIN_SIDE_PX or h < MIN_SIDE_PX:
                return sample_id, False, "too_small"
            im.save(path, format="JPEG", quality=90)
        return sample_id, True, "downloaded"
    except Exception as e:
        if os.path.exists(path):
            try:
                os.remove(path)
            except OSError:
                pass
        return sample_id, False, type(e).__name__


def download_split(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = list(zip(df[ID_COL].astype(str), df[URL_COL].astype(str)))
    ok_ids = set()
    stats = {"cached": 0, "downloaded": 0, "failed": 0}

    with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_one, sid, url) for sid, url in rows]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"download:{split_name}"):
            sid, ok, reason = fut.result()
            if ok:
                ok_ids.add(sid)
                stats["cached" if reason == "cached" else "downloaded"] += 1
            else:
                stats["failed"] += 1

    kept = df[df[ID_COL].astype(str).isin(ok_ids)].copy().reset_index(drop=True)
    kept["image_path"] = kept[ID_COL].astype(str).map(cache_path_for)
    print(f"{split_name}: kept={len(kept):,}  stats={stats}")
    return kept


train_img = download_split(train_df, "train")
val_img = download_split(val_df, "val")
test_img = download_split(test_df, "test")

assert len(train_img) > 0, "No training images downloaded — check network / URLs."
assert len(val_img) > 0 and len(test_img) > 0, "Val/test image sets are empty."
print("Ready for training.")


## Preprocessing and dataloaders

Train transforms: resize → random crop / flip / light color jitter → ImageNet normalize.  
Eval transforms: resize → center crop → ImageNet normalize (same mean/std as the ResNet50 backbone).


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMG_SIZE = 224
BATCH_SIZE = 32 if DEVICE == "cuda" else 8
NUM_WORKERS = 2


train_tf = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

eval_tf = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)


class FakedditImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        path = row["image_path"]
        with Image.open(path) as im:
            image = im.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = int(row[LABEL_COL])
        return image, label


train_ds = FakedditImageDataset(train_img, train_tf)
val_ds = FakedditImageDataset(val_img, eval_tf)
test_ds = FakedditImageDataset(test_img, eval_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda")
)

print(f"batches → train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}")


## Model — ResNet50 (frozen stem + early layers)

Load ImageNet weights. Freeze all parameters, then unfreeze `layer4` and the final fully connected head for binary classification.


In [ ]:
def build_resnet50(num_classes: int = 2) -> nn.Module:
    # torchvision API: Weights enum (new) or pretrained=True (old)
    try:
        from torchvision.models import ResNet50_Weights

        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    except Exception:
        model = resnet50(pretrained=True)

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    # Freeze entire backbone first
    for p in model.parameters():
        p.requires_grad = False

    # Fine-tune last residual block + classifier
    for p in model.layer4.parameters():
        p.requires_grad = True
    for p in model.fc.parameters():
        p.requires_grad = True

    return model


model = build_resnet50(num_classes=2).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")


## Train

Cross-entropy with class weights (fake/real skew). AdamW on trainable parameters only. Track validation macro-F1 and keep the best checkpoint in memory.


In [ ]:
EPOCHS = 4
LR = 1e-4
WEIGHT_DECAY = 1e-4

# Class weights from the downloaded train set
counts = train_img[LABEL_COL].value_counts().sort_index()
n_total = float(counts.sum())
class_weights = torch.tensor(
    [n_total / (2.0 * float(counts.get(0, 1))), n_total / (2.0 * float(counts.get(1, 1)))],
    dtype=torch.float32,
    device=DEVICE,
)
print("class_weights:", class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
# torch.amp (new) or torch.cuda.amp (older Colab images)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    def autocast_ctx():
        return torch.amp.autocast("cuda", enabled=(DEVICE == "cuda"))
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    def autocast_ctx():
        return torch.cuda.amp.autocast(enabled=(DEVICE == "cuda"))


@torch.no_grad()
def predict_loader(loader: DataLoader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        with autocast_ctx():
            logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.numpy())
    return np.concatenate(all_labels), np.concatenate(all_preds)


def run_epoch(loader: DataLoader, train: bool) -> float:
    model.train(train)
    total_loss, n = 0.0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast_ctx():
                logits = model(images)
                loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += float(loss.item()) * labels.size(0)
            n += labels.size(0)
    return total_loss / max(n, 1)


best_state = None
best_val_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, train=True)
    y_true, y_pred = predict_loader(val_loader)
    val_acc = accuracy_score(y_true, y_pred)
    val_f1 = f1_score(y_true, y_pred, average="macro")
    history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_acc": val_acc, "val_f1_macro": val_f1}
    )
    print(
        f"Epoch {epoch}/{EPOCHS}  loss={train_loss:.4f}  "
        f"val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({time.time() - t0:.1f}s)"
    )
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ↑ new best val macro-F1: {best_val_f1:.4f}")

assert best_state is not None
model.load_state_dict(best_state)
model.to(DEVICE)
print("Restored best validation checkpoint.")


## Evaluate

Report accuracy and per-class F1 on the image-available validation and test subsets (not the full TSV rows that failed download).


In [ ]:
LABEL_NAMES = ["fake", "real"]


def evaluate_classifier(name: str, y_true, y_pred) -> dict:
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")
    print(f"\n===== {name} =====")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 macro: {f1_macro:.4f} | F1 weighted: {f1_weighted:.4f}")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=LABEL_NAMES,
        yticklabels=LABEL_NAMES,
        ax=ax,
    )
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(name)
    plt.tight_layout()
    plt.show()
    return {"accuracy": acc, "f1_macro": f1_macro, "f1_weighted": f1_weighted}


y_val, pred_val = predict_loader(val_loader)
y_test, pred_test = predict_loader(test_loader)

resnet_val_metrics = evaluate_classifier("ResNet50 — VAL (image-available)", y_val, pred_val)
resnet_test_metrics = evaluate_classifier("ResNet50 — TEST (image-available)", y_test, pred_test)


## Save checkpoint

Persist `state_dict` plus a small JSON of training config under `DATA_DIR/checkpoints/module02_resnet50/` for fusion / demo.


In [ ]:
import json

SAVE_DIR = os.path.join(DATA_DIR, "checkpoints", "module02_resnet50")
os.makedirs(SAVE_DIR, exist_ok=True)

ckpt_path = os.path.join(SAVE_DIR, "model.pt")
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "arch": "resnet50",
        "num_classes": 2,
        "imagenet_mean": IMAGENET_MEAN,
        "imagenet_std": IMAGENET_STD,
        "img_size": IMG_SIZE,
        "label_names": LABEL_NAMES,
        "best_val_f1_macro": best_val_f1,
        "history": history,
        "train_size": len(train_img),
        "val_size": len(val_img),
        "test_size": len(test_img),
    },
    ckpt_path,
)

meta = {
    "arch": "resnet50",
    "freeze": "all except layer4 + fc",
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "subsample_train": SUBSAMPLE_TRAIN,
    "subsample_val": SUBSAMPLE_VAL,
    "subsample_test": SUBSAMPLE_TEST,
    "best_val_f1_macro": best_val_f1,
    "metrics_val": resnet_val_metrics,
    "metrics_test": resnet_test_metrics,
}
with open(os.path.join(SAVE_DIR, "run_config.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print(f"Saved checkpoint → {ckpt_path}")


## Explainability — Grad-CAM

Target the **last convolutional block** (`layer4[-1]`), not the classifier. Overlay the activation map on a held-out test image.

> **Note.** Faithfulness (blanking the highlighted region and re-scoring) is Module 04.


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image


def denormalize(tensor_chw: torch.Tensor) -> np.ndarray:
    """CHW ImageNet-normalized tensor → HWC float RGB in [0, 1]."""
    img = tensor_chw.detach().cpu().clone()
    for t, m, s in zip(img, IMAGENET_MEAN, IMAGENET_STD):
        t.mul_(s).add_(m)
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return img


# Grad-CAM must target the last conv layer
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

# One test example (prefer a fake-labeled row when available)
fake_rows = test_img[test_img[LABEL_COL] == 0]
example_row = fake_rows.iloc[0] if len(fake_rows) else test_img.iloc[0]
example_path = example_row["image_path"]
example_true = int(example_row[LABEL_COL])

with Image.open(example_path) as im:
    pil_img = im.convert("RGB")

input_tensor = eval_tf(pil_img).unsqueeze(0).to(DEVICE)
model.eval()
with torch.no_grad():
    logits = model(input_tensor)
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred = int(logits.argmax(dim=1).item())

print(f"id={example_row[ID_COL]}  true={LABEL_NAMES[example_true]}  "
      f"pred={LABEL_NAMES[pred]}  proba_fake={probs[0]:.4f}  proba_real={probs[1]:.4f}")

# Explain the predicted class
targets = [ClassifierOutputTarget(pred)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

rgb = denormalize(input_tensor[0])
overlay = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(pil_img.resize((IMG_SIZE, IMG_SIZE)))
axes[0].set_title("original")
axes[1].imshow(grayscale_cam, cmap="jet")
axes[1].set_title("Grad-CAM")
axes[2].imshow(overlay)
axes[2].set_title(f"overlay → {LABEL_NAMES[pred]}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

# Persist before show (Colab may clear the figure after display)
cam_out = os.path.join(SAVE_DIR, "gradcam_example.png")
fig.savefig(cam_out, dpi=150, bbox_inches="tight")
print(f"Wrote {cam_out}")
plt.show()


## Results

Compare against Module 01 text metrics when writing the ablation table. Image scores are on the **successfully downloaded** subset only — document that in the report.


In [ ]:
summary = pd.DataFrame(
    [
        {"model": "ResNet50", "split": "test", **resnet_test_metrics},
        {"model": "ResNet50", "split": "val", **resnet_val_metrics},
    ]
)
display(summary)

# Optional: join Module 01 CSV if present on Drive
m1_path = os.path.join(DATA_DIR, "checkpoints", "module01_metrics.csv")
if os.path.exists(m1_path):
    m1 = pd.read_csv(m1_path)
    print("\nModule 01 (text) metrics for reference:")
    display(m1)
else:
    print("Module 01 metrics CSV not found on Drive (optional).")

summary_path = os.path.join(DATA_DIR, "checkpoints", "module02_metrics.csv")
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
summary.to_csv(summary_path, index=False)
print(f"Wrote {summary_path}")


## Artifacts

| Output | Location |
|--------|----------|
| ResNet50 `state_dict` + meta | `{DATA_DIR}/checkpoints/module02_resnet50/model.pt` |
| Run config JSON | `{DATA_DIR}/checkpoints/module02_resnet50/run_config.json` |
| Grad-CAM example PNG | `{DATA_DIR}/checkpoints/module02_resnet50/gradcam_example.png` |
| Metrics table | `{DATA_DIR}/checkpoints/module02_metrics.csv` |
| Image cache | `{DATA_DIR}/image_cache/` |

These scores are the **image-only** entries in the project ablation table. Next: **Module 03** — late fusion of DistilBERT + ResNet embeddings and a text–image agreement explanation.

**Hosted-GPU tips.** Lower `SUBSAMPLE_*` or `BATCH_SIZE` if you OOM. Reuse `image_cache` across sessions. Do not commit `image_cache/` or `model.pt` to git — keep them on Drive (same pattern as Module 01).
